In [44]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
import scipy
from hmmlearn import hmm
from sklearn.preprocessing import StandardScaler

data_filled = pd.read_csv('../data/processed/chip_chain_processed.csv')
data_daily_returns = pd.read_csv('../data/processed/chip_chain_daily_returns.csv')
data_log_returns = pd.read_csv('../data/processed/chip_chain_log_returns.csv')
data_indicators = pd.read_csv('../data/processed/chip_chain_custom_indicators.csv')

Mainteanant que l'on a une partie solide essayons d'implémenter du machine learning (ou plus précisément deep learning), le problème c'est que l'on a pas de dataset solide pour avoir toute les donnes notamment liés aux hmm sans dataleakage

Ainsi nous allons regrouper sous un dataset :
- les logs_returs
- indicators
- les états cachés (on recalcul pour ne pas avoir de problème on veut un 80% Viterbi pour train et 20% Forward pour test et en le faisant un par un)

In [45]:
# Aligner la data
data_log_returns = data_log_returns.iloc[62:].reset_index(drop=True)
data_indicators = data_indicators.reset_index(drop=True)

# Préparer les features pour le HMM
global_log_returns = data_log_returns.drop(columns=['Date', 'GLD', 'TLT', '^TNX', '^VIX']).mean(axis=1)
gld_tlt_log_returns = data_log_returns[['GLD', 'TLT']].mean(axis=1)

hmm_features = pd.DataFrame({
    'ChipFearIndex': data_indicators['Chip_Fear_Index'],
    'GlobalLogReturns': global_log_returns,
    'GLD_TLT_LogReturns': gld_tlt_log_returns,
    'TNX_Diff': data_indicators['TNX_Diff']
})

# Découpage 80/20 strict
split_index = int(len(hmm_features) * 0.8)
X_train_raw = hmm_features.iloc[:split_index].values
X_test_raw = hmm_features.iloc[split_index:].values

# Standardisation sans data leakage
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)    # on ne fit pas sur le test set pour éviter le data leakage, on utilise les paramètres du train set pour transformer le test set

# Entrainement du hmm (uniquement sur le train set)
model_hmm = hmm.GaussianHMM(n_components=3, covariance_type="full", n_iter=1000, random_state=42)
model_hmm.fit(X_train_scaled)

# Calcul des états train avec Viterbi
train_states = model_hmm.predict(X_train_scaled)

# on applique le renommage des états pour rester cohérent avec le notebook 06_advanced_portfolio.ipynb (0<=>1, 2 rest 2)
train_states_renamed = [1 if state == 0 else 0 if state == 1 else 2 for state in train_states]

# Calcul des états test avec forward algorithm (on ne refit pas le modèle sur le test set pour éviter le data leakage)
test_states = []

X_full_scaled = np.vstack((X_train_scaled, X_test_scaled))  # on combine avoir aussi tout en meme temps

# on doit dévoiler jour après jour l'historique pour le forward algorithm, donc on va faire un loop sur le test set
for i in range(len(X_test_scaled)):
    X_visible = X_full_scaled[:split_index + i + 1]  # on prend tout le train set + le test set jusqu'à l'index i
    probs = model_hmm.predict_proba(X_visible)
    current_state = np.argmax(probs[-1])  # on prend l'état le plus probable pour le dernier jour visible
    mapped_state = 1 if current_state == 0 else 0 if current_state == 1 else 2
    test_states.append(mapped_state)

# Création du dataset final pour le LSTM
master_df = data_log_returns.copy()

for col in data_indicators.columns:
    if col != 'Date':
        master_df[col] = data_indicators[col]

all_states = np.concatenate((train_states_renamed, test_states))
master_df['HMM_State'] = all_states

# on ajoute une colonne drapeau pour filtrer les données du train set et du test set
master_df['Dataset_Type'] = ['Train'] * split_index + ['Test'] * (len(X_test_scaled))

outputs_path = '../data/processed/chip_chain_master_lstm.csv'
master_df.to_csv(outputs_path, index=False)

Mainteant nous pouvons utiliser la data pour travailler sur le LSTM, il reste néanmoins un détail le HMM est défini 0, 1 ou 2 mais on ne veut pas de lien ou 1 est le milieu donc on encode one-hot

In [46]:
master_df = pd.get_dummies(master_df, columns=['HMM_State'], prefix='HMM_State', dtype=int)  # one-hot encoding des états HMM pour le LSTM

On souhaite pour l'instant ce concentrer sur une data pour voir les effets, on choisit NVDA car elle n'est pas en début de supply chain cela pemmetrait de voir les impacts des relations (on espère ASML=>TSM=>NVDA)

In [47]:
TARGET_ASSET = 'NVDA'
master_df['Target'] = np.where(master_df[TARGET_ASSET] > 0, 1, 0) # Convert to binary returns (1 for positive return, 0 for negative return)

# la dernière data n'a pas de demain, donc on la supprime
master_df.dropna(inplace=True)

On ne s'intéresse pas à combien il augmente car sinon les réponses possibles sont trop nombreuses, alors on veut juste savoir si ça à fait plus ou moins par rapport à hier

Néanmoins le test de Ljung-Box a montré que l'autocorrélation était bien plus importante pour des lag 5 ou 21. C'est pourquoi on choisit un time step de 21, cela signifie que pour prédire le jour T+1, le LSTM va intégrer un bloc contenant tout les jours de T-20 jusqu'à T, ainsi il pourra sûrement voir la réaction entre ASML à T-15 puis TSM à T-5 et donc NVDA à T+1. (Attention le test de Ljung-Box est que sur de l'autocorrélation, il montre un certaine inertie mais par contre si on veut montrer la corrélation entre deux différents ont peut faire le test de Causalité de Granger, mais bon ici on veut aussi que le LSTM trouve tout seul les relations)

In [48]:
TIME_STEPS = 21

Ensuite :
- on sépare la data en train et test comme prévu au dessus (80/20)
- on supprime les colonnes sans réelles informations sur le mouvement
- on standardise

In [49]:
# on sépare le train set et le test set pour le LSTM
train_mask = master_df['Dataset_Type'] == 'Train'
test_mask = master_df['Dataset_Type'] == 'Test'

# on supprime les colonnes qui ne sont pas des features pour le LSTM
feature_columns = [col for col in master_df.columns if col not in ['Date', 'Dataset_Type', 'Target']]

# on standardise les features pour le LSTM
scaler_lstm = StandardScaler()

train_scaled = scaler_lstm.fit_transform(master_df.loc[train_mask, feature_columns])
test_scaled = scaler_lstm.transform(master_df.loc[test_mask, feature_columns])  # on ne fit pas sur le test set pour éviter le data leakage, on utilise les paramètres du train set pour transformer le test set

X_scaled_full = np.vstack((train_scaled, test_scaled))
y_full = master_df['Target'].values
type_full = master_df['Dataset_Type'].values

On créer des tenseurs 3D (sliding window)

In [50]:
print('On travaille sur un time step de {}'.format(TIME_STEPS))
X_3d_train, y_3d_train = [], []
X_3d_test, y_3d_test = [], []

for i in range(len(X_scaled_full) - TIME_STEPS):
    window = X_scaled_full[i:i + TIME_STEPS]
    target_value = y_full[i + TIME_STEPS]
    dataset_type_of_the_day = type_full[i + TIME_STEPS]

    if dataset_type_of_the_day == 'Train':
        X_3d_train.append(window)
        y_3d_train.append(target_value)
    else:
        X_3d_test.append(window)
        y_3d_test.append(target_value)

# on convertit les listes en numpy arrays
X_train_3d = np.array(X_3d_train)
y_train = np.array(y_3d_train)
X_test_3d = np.array(X_3d_test)
y_test = np.array(y_3d_test)

On travaille sur un time step de 21


Pour fini on valide que tout est bon

In [51]:
print(f"Nombre de Features ingérées : {len(feature_columns)}")
print("-" * 30)
print(f"Shape X_train 3D : {X_train_3d.shape}")
print(f"Shape y_train    : {y_train.shape}")
print("-" * 30)
print(f"Shape X_test 3D  : {X_test_3d.shape}")
print(f"Shape y_test     : {y_test.shape}")

Nombre de Features ingérées : 51
------------------------------
Shape X_train 3D : (2804, 21, 51)
Shape y_train    : (2804,)
------------------------------
Shape X_test 3D  : (707, 21, 51)
Shape y_test     : (707,)


On sauvegarde les tenseurs pour les utiliser à la séance suivante

In [52]:
os.makedirs('../data/tensors', exist_ok=True)

tensors_path = '../data/tensors/lstm_tensors.npz'

np.savez_compressed(
    tensors_path,
    X_train=X_train_3d,
    y_train=y_train,
    X_test=X_test_3d,
    y_test=y_test
)

print(f" Tenseurs 3D sauvegardés avec succès dans : {tensors_path}")

 Tenseurs 3D sauvegardés avec succès dans : ../data/tensors/lstm_tensors.npz
